## Safe 2 pairs

- Take a directory with a bunch of safe files, extract the dates, 
- Given the date/time of the earthquake, generate a list of coseismic, pre-seismic, and post-seismic pairs

In [7]:
from pathlib import Path
from datetime import datetime
from itertools import combinations

# --------------------------------------------------
# Earthquake time
# --------------------------------------------------

event_time = datetime(2026, 4, 14, 1, 29, 12)

# --------------------------------------------------
# Read SAFE files
# --------------------------------------------------

safedir = '/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_phasegrad/raw/'

## Make a list of scenes
scenes = []
for path in Path(safedir).rglob("S1*.SAFE"):
    datestr = path.stem.split('_')[5]
    dt = datetime.strptime(
        datestr,
        "%Y%m%dT%H%M%S"
    )
    scenes.append({
        "datetime": dt,
        "path": path,
        "safe_name": path.name
    })
# Sort by acquisition time
scenes = sorted(
    scenes,
    key=lambda x: x["datetime"]
)

eofs = []

for path in Path(safedir).rglob("S1*.EOF"):

    parts = path.stem.split('_')

    validity_start = datetime.strptime(
        parts[-2][1:],   # remove leading "V"
        "%Y%m%dT%H%M%S"
    )

    validity_stop = datetime.strptime(
        parts[-1],
        "%Y%m%dT%H%M%S"
    )

    eofs.append({
        "path": path,
        "name": path.name,
        "start": validity_start,
        "stop": validity_stop
    })

for scene in scenes:

    acquisition_time = scene["datetime"]

    matching_eof = None

    for eof in eofs:

        if eof["start"] <= acquisition_time <= eof["stop"]:

            matching_eof = eof
            break

    if matching_eof is None:

        raise ValueError(
            f"No EOF found for {scene['safe_name']}"
        )

    scene["eof_name"] = matching_eof["name"]

import pandas as pd
df_scenes = pd.DataFrame(scenes)
df_scenes

,datetime,path,safe_name,eof_name
0,2026-04-04 01:50:53,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,S1C_IW_SLC__1SSV_20260404T015053_20260404T0151...,S1C_OPER_AUX_POEORB_OPOD_20260424T070857_V2026...
1,2026-04-10 01:51:52,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,S1A_IW_SLC__1SSV_20260410T015152_20260410T0152...,S1A_OPER_AUX_POEORB_OPOD_20260430T070535_V2026...
2,2026-04-16 01:50:54,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,S1C_IW_SLC__1SSV_20260416T015054_20260416T0151...,S1C_OPER_AUX_POEORB_OPOD_20260506T070925_V2026...
3,2026-04-22 01:51:52,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,S1A_IW_SLC__1SSV_20260422T015152_20260422T0152...,S1A_OPER_AUX_POEORB_OPOD_20260512T070315_V2026...
4,2026-04-28 01:50:54,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,S1C_IW_SLC__1SSV_20260428T015054_20260428T0151...,S1C_OPER_AUX_POEORB_OPOD_20260518T070752_V2026...
5,2026-05-04 01:51:51,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,S1A_IW_SLC__1SSV_20260504T015151_20260504T0152...,S1A_OPER_AUX_POEORB_OPOD_20260524T070443_V2026...
6,2026-05-16 01:51:51,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,S1A_IW_SLC__1SSV_20260516T015151_20260516T0151...,S1A_OPER_AUX_RESORB_OPOD_20260516T052718_V2026...


In [8]:
## Make a list of scene pairs
pairs = []

for s1, s2 in combinations(scenes, 2):

    d1 = s1["datetime"]
    d2 = s2["datetime"]

    pair = {
        "reference_datetime": d1,
        "repeat_datetime": d2,

        "reference_safe": s1["safe_name"],
        "reference_eof": s1["eof_name"],
        "repeat_safe": s2["safe_name"],
        "repeat_eof": s2["eof_name"],

        "reference_path": s1["path"],
        "repeat_path": s2["path"],

        "baseline_days": (d2 - d1).days,

        # YYYYDOY (GMTSAR-style DOY offset)
        "ref_doy": f"{d1.year}{d1.timetuple().tm_yday - 1:03d}",
        "rep_doy": f"{d2.year}{d2.timetuple().tm_yday - 1:03d}",

        # YYYYMMDD
        "ref_date": d1.strftime("%Y%m%d"),
        "rep_date": d2.strftime("%Y%m%d"),
    }

    # Categorize relative to earthquake
    if d2 < event_time:
        pair["category"] = "preseismic"
    elif d1 < event_time < d2:
        pair["category"] = "coseismic"
    elif d1 > event_time:
        pair["category"] = "postseismic"

    pairs.append(pair)

pairs_df = pd.DataFrame(pairs)
pairs_df.to_csv(f"{safedir}pairs.csv", index=False)

pairs_df

,reference_datetime,repeat_datetime,reference_safe,reference_eof,repeat_safe,repeat_eof,reference_path,repeat_path,baseline_days,ref_doy,rep_doy,ref_date,rep_date,category
0,2026-04-04 01:50:53,2026-04-10 01:51:52,S1C_IW_SLC__1SSV_20260404T015053_20260404T0151...,S1C_OPER_AUX_POEORB_OPOD_20260424T070857_V2026...,S1A_IW_SLC__1SSV_20260410T015152_20260410T0152...,S1A_OPER_AUX_POEORB_OPOD_20260430T070535_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,6,2026093,2026099,20260404,20260410,preseismic
1,2026-04-04 01:50:53,2026-04-16 01:50:54,S1C_IW_SLC__1SSV_20260404T015053_20260404T0151...,S1C_OPER_AUX_POEORB_OPOD_20260424T070857_V2026...,S1C_IW_SLC__1SSV_20260416T015054_20260416T0151...,S1C_OPER_AUX_POEORB_OPOD_20260506T070925_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,12,2026093,2026105,20260404,20260416,coseismic
2,2026-04-04 01:50:53,2026-04-22 01:51:52,S1C_IW_SLC__1SSV_20260404T015053_20260404T0151...,S1C_OPER_AUX_POEORB_OPOD_20260424T070857_V2026...,S1A_IW_SLC__1SSV_20260422T015152_20260422T0152...,S1A_OPER_AUX_POEORB_OPOD_20260512T070315_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,18,2026093,2026111,20260404,20260422,coseismic
3,2026-04-04 01:50:53,2026-04-28 01:50:54,S1C_IW_SLC__1SSV_20260404T015053_20260404T0151...,S1C_OPER_AUX_POEORB_OPOD_20260424T070857_V2026...,S1C_IW_SLC__1SSV_20260428T015054_20260428T0151...,S1C_OPER_AUX_POEORB_OPOD_20260518T070752_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,24,2026093,2026117,20260404,20260428,coseismic
4,2026-04-04 01:50:53,2026-05-04 01:51:51,S1C_IW_SLC__1SSV_20260404T015053_20260404T0151...,S1C_OPER_AUX_POEORB_OPOD_20260424T070857_V2026...,S1A_IW_SLC__1SSV_20260504T015151_20260504T0152...,S1A_OPER_AUX_POEORB_OPOD_20260524T070443_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,30,2026093,2026123,20260404,20260504,coseismic
5,2026-04-04 01:50:53,2026-05-16 01:51:51,S1C_IW_SLC__1SSV_20260404T015053_20260404T0151...,S1C_OPER_AUX_POEORB_OPOD_20260424T070857_V2026...,S1A_IW_SLC__1SSV_20260516T015151_20260516T0151...,S1A_OPER_AUX_RESORB_OPOD_20260516T052718_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,42,2026093,2026135,20260404,20260516,coseismic
6,2026-04-10 01:51:52,2026-04-16 01:50:54,S1A_IW_SLC__1SSV_20260410T015152_20260410T0152...,S1A_OPER_AUX_POEORB_OPOD_20260430T070535_V2026...,S1C_IW_SLC__1SSV_20260416T015054_20260416T0151...,S1C_OPER_AUX_POEORB_OPOD_20260506T070925_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,5,2026099,2026105,20260410,20260416,coseismic
7,2026-04-10 01:51:52,2026-04-22 01:51:52,S1A_IW_SLC__1SSV_20260410T015152_20260410T0152...,S1A_OPER_AUX_POEORB_OPOD_20260430T070535_V2026...,S1A_IW_SLC__1SSV_20260422T015152_20260422T0152...,S1A_OPER_AUX_POEORB_OPOD_20260512T070315_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,12,2026099,2026111,20260410,20260422,coseismic
8,2026-04-10 01:51:52,2026-04-28 01:50:54,S1A_IW_SLC__1SSV_20260410T015152_20260410T0152...,S1A_OPER_AUX_POEORB_OPOD_20260430T070535_V2026...,S1C_IW_SLC__1SSV_20260428T015054_20260428T0151...,S1C_OPER_AUX_POEORB_OPOD_20260518T070752_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,17,2026099,2026117,20260410,20260428,coseismic
9,2026-04-10 01:51:52,2026-05-04 01:51:51,S1A_IW_SLC__1SSV_20260410T015152_20260410T0152...,S1A_OPER_AUX_POEORB_OPOD_20260430T070535_V2026...,S1A_IW_SLC__1SSV_20260504T015151_20260504T0152...,S1A_OPER_AUX_POEORB_OPOD_20260524T070443_V2026...,/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_ph...,/Volumes/T9_InSAR/2026-04-14_nevada/

## Generate run files for each pair


In [9]:
categories = {
    "preseismic": [],
    "coseismic": [],
    "postseismic": []
}

for pair in pairs:
    categories[pair["category"]].append(pair)

from pathlib import Path

def write_pair_script(pair_list, output_file):

    with open(output_file, "w") as f:

        # Bash header
        f.write("#!/bin/bash\n\n")

        for i, pair in enumerate(pair_list, start=1):

            safefile1 = pair["reference_safe"]
            eoffile1 = pair["reference_eof"]

            safefile2 = pair["repeat_safe"]
            eoffile2 = pair["repeat_eof"]

            pair_doystring = (f"{pair['ref_doy']}_{pair['rep_doy']}")
            
            logfile = (
                f"log_"
                f"{pair['reference_datetime'].strftime('%Y%m%d')}_"
                f"{pair['repeat_datetime'].strftime('%Y%m%d')}.txt"
            )

            command = (
                f"p2p_S1_TOPS_Frame.csh "
                f"{safefile1} {eoffile1} "
                f"{safefile2} {eoffile2} "
                f"config.txt vv 1 "
                f">& {logfile}\n\n"
                f"mv {logfile} merge/{logfile} \n"
                f"cp config.txt merge/config.txt \n"
                f"python /Users/hyin/soft/insar_surface-def_tools/grd2geotiff.py merge/ \n"
                f"mv merge merge_{pair_doystring}_unwrap \n"
                f"rm -rf F1 F2 F3 \n\n"
                # f"cp F2/topo/trans.dat F2/intf/{pair_doystring}/ \n"
                # f"cp F3/topo/trans.dat F3/intf/{pair_doystring}/ \n\n"
            )


            f.write(command)
            #                 f"Ref DOY: {pair['ref_doy']}"

    print(f"Wrote {output_file}")


# pathdir = '/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_gmtsar_stack/'
write_pair_script(
    categories["preseismic"],
    f"{safedir}preseismic_pairs.sh"
)

write_pair_script(
    categories["coseismic"],
    f"{safedir}coseismic_pairs.sh"
)

write_pair_script(
    categories["postseismic"],
    f"{safedir}postseismic_pairs.sh"
)

Wrote /Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_phasegrad/raw/preseismic_pairs.sh
Wrote /Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_phasegrad/raw/coseismic_pairs.sh
Wrote /Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_phasegrad/raw/postseismic_pairs.sh


## Make a pairwise timeline